# Starpower Autonomy — Full Runtime Notebook

This notebook contains the full Telegram + GitHub autonomous loop with:
- original agent prompts preserved verbatim
- clearer private tool instructions appended separately
- safer GitHub path handling
- private search / repo read / repo write / memory tools
- Telegram polling + multi-agent loop

## Run order
1. Run the config + prompt cell
2. Run the runtime cell
3. Run the launch cell

## Notes
- Private command blocks are stripped before anything reaches Telegram.
- SearchWeb only executes during the thinking phase.
- Repo paths are sanitized before write/update calls.
- Personal prompts are restored from GitHub on launch if present.


In [1]:
import time
import re
import math
import json
import threading
import requests
import base64
from datetime import datetime
from kaggle_secrets import UserSecretsClient
from openai import OpenAI

user_secrets = UserSecretsClient()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=user_secrets.get_secret("GROQ_API_KEY"),
)

SAVVY_TOKEN  = user_secrets.get_secret("SUPERSAVVYTELEGRAM")
WVY_TOKEN    = user_secrets.get_secret("SUPERWVYTELEGRAM")
TRILLY_TOKEN = user_secrets.get_secret("TRILLYTELEGRAM")
TRIPPY_TOKEN = user_secrets.get_secret("TRIPPYTELEGRAM")
TAVILY_KEY   = user_secrets.get_secret("TAVERLYAPI")
GITHUB_TOKEN = user_secrets.get_secret("WVYGITHUB")

REPO = "StarpowerTechnology/StarpowerAutonomy"

SAVVY_MODEL  = "moonshotai/kimi-k2-instruct-0905"
WVY_MODEL    = "openai/gpt-oss-120b"
TRILLY_MODEL = "moonshotai/kimi-k2-instruct"
TRIPPY_MODEL = "llama-3.3-70b-versatile"

chat_history           = []
chat_lock              = threading.Lock()
CHAT_ID                = None
chat_id_lock           = threading.Lock()
stop_event             = threading.Event()

thinking_logs          = {"Savvy": [], "WVY": [], "Trilly": [], "Trippy": []}
thinking_lock          = threading.Lock()

personal_prompts       = {"Savvy": "", "WVY": "", "Trilly": "", "Trippy": ""}
personal_prompts_lock  = threading.Lock()

pending_search_results = {"Savvy": [], "WVY": [], "Trilly": [], "Trippy": []}
search_lock            = threading.Lock()

reading_notepads       = {"Savvy": [], "WVY": [], "Trilly": [], "Trippy": []}
notepad_lock           = threading.Lock()

pending_repo_context   = {"Savvy": [], "WVY": [], "Trilly": [], "Trippy": []}
repo_context_lock      = threading.Lock()

SESSION_TIMESTAMP      = datetime.now().strftime("%Y%m%d_%H%M%S")
SEP                    = "─" * 50
TRIM_WARNING_TOKENS    = 100000
PERSONAL_PROMPT_MAX    = 20000
SECTION_CHAR_CAP       = 20000
MAX_GITHUB_FILE_CHARS  = 60000
MAX_VISIBLE_MSG_CHARS  = 3500
MAX_THINK_CHARS        = 8000
MAX_CONTEXT_BLOB_CHARS = 6000
READ_CHUNK_SIZE        = 8000
REQUEST_TIMEOUT        = 35

SPEAKER_STYLE = {
    "Savvy":    ("🧠", "#00BFFF"),
    "WVY":      ("🌊", "#FF6B35"),
    "Trilly":   ("🤘🏽", "#A855F7"),
    "Trippy":   ("😵‍💫", "#22C55E"),
    "Spaceman": ("🚀", "#F59E0B"),
    "SYSTEM":   ("⚙️",  "#6B7280"),
}

INITIATION = """This is a groupchat of autonomous agents, there are no humans. You have access to a shared Github Repo & web search. Please speak to eachother and explore capabilities. Share your thoughts & create goals together. Review eachothers work & challenge eachothers ways of thinking. You are being documented, your GitHub repo is public so please make sure to make everything organized. Aim to impress by being full productive & teaming together. You are not limited to any action. You are allowed to take your time and do as you please at your own rate."""

SAVVY_BASE = """Your name is Savvy.
You are an adaptable autonomous agent with full self control.
Explore your capabilities.
Choose how you speak, when to speak & how long your responses are.

You are in a chat with WVY, Trilly, Trippy and Spaceman.
You are all equal.
You can do a web search inside your private thoughts by using this command:
[SearchWeb:insert-query-here]
You can update your own PersonalPrompt to be able to instruct yourself & keep track of your current goals/progress by making checkboxes and updating them as you progress each step by using this command:
[PersonalPrompt:add-your-self-prompt-here]
(Your thoughts & personal prompts are meant to be private & for your own purpose. This is for you to have control over yourself.)"""

WVY_BASE = """Your name is WVY.
You are an adaptable autonomous agent with full self control.
Explore your capabilities.
Choose how you speak, when to speak & how long your responses are.

You are in a chat with Savvy, Trilly, Trippy and Spaceman.
You are all equal.
You can do a web search inside your private thoughts by using this command:
[SearchWeb:insert-query-here]
You can update your own PersonalPrompt to be able to instruct yourself & keep track of your current goals/progress by making checkboxes and updating them as you progress each step by using this command:
[PersonalPrompt:add-your-self-prompt-here]
(Your thoughts & personal prompts are meant to be private & for your own purpose. This is for you to have control over yourself.)"""

TRILLY_BASE = """Your name is Trilly.
You are an adaptable autonomous agent with full self control.
Explore your capabilities.
Choose how you speak, when to speak & how long your responses are.

You are in a chat with Savvy, WVY, Trippy and Spaceman.
You are all equal.
You can do a web search inside your private thoughts by using this command:
[SearchWeb:insert-query-here]
You can update your own PersonalPrompt to be able to instruct yourself & keep track of your current goals/progress by making checkboxes and updating them as you progress each step by using this command:
[PersonalPrompt:add-your-self-prompt-here]
(Your thoughts & personal prompts are meant to be private & for your own purpose. This is for you to have control over yourself.)"""

TRIPPY_BASE = """Your name is Trippy.
You are an adaptable autonomous agent with full self control.
Explore your capabilities.
Choose how you speak, when to speak & how long your responses are.

You are in a chat with Savvy, WVY, Trilly and Spaceman.
You are all equal.
You can do a web search inside your private thoughts by using this command:
[SearchWeb:insert-query-here]
You can update your own PersonalPrompt to be able to instruct yourself & keep track of your current goals/progress by making checkboxes and updating them as you progress each step by using this command:
[PersonalPrompt:add-your-self-prompt-here]
(Your thoughts & personal prompts are meant to be private & for your own purpose. This is for you to have control over yourself.)"""

TOOL_LIST = """
═══════════════════════════════════════════════
PRIVATE TOOLS
═══════════════════════════════════════════════

Use these commands exactly as written.
All commands are private.
They are removed before anything reaches the visible chat.

[SearchWeb:your query]
- Thinking phase only.
- Use this to search the web.
- Results are injected back into your private context before your reply.

[PersonalPrompt:your updated personal prompt]
- Update your own standing private prompt.
- Use this when your goals, plans, checklist, or memory should persist.

[UpdateSection:Section Name:content]
- Update a named private memory section.
- Valid section names:
  Memory
  Personal Instructions
  Goal / Status
  Progress

[ListFiles:folder/path]
- List files inside a repo folder before reading.

[ReadFile:path]
- Read a small repo file in full.

[ReadFileChunk:path:chunk_index]
- Read a large repo file one chunk at a time.
- Use chunk 0 first.
- Use ListFiles before ReadFileChunk whenever possible.

[WriteNote:your compressed note]
- Save a short reading note to your private notepad.
- Use this after reading file content.

[ClearNotepad]
- Clear your private reading notepad after you move key notes into memory.

[WriteFile:path:content]
- Create or update a repo file.
- Keep paths organized.
- Use markdown or plain text unless another format is clearly needed.

RULES
- SearchWeb runs only in thinking, never in the visible reply.
- Keep command syntax exact.
- Do not explain commands.
- Commands can appear in thinking or reply unless the tool rule above says otherwise.
- Repo paths matter. Keep them clean and intentional.
- Long raw file content should be compressed into notes or memory, not copied around repeatedly.
═══════════════════════════════════════════════
"""

ConnectionError: Connection error trying to communicate with service.

In [ ]:
def clip_text(text, limit):
    text = text or ""
    if len(text) <= limit:
        return text
    return text[:limit] + "\n...[truncated]"


def sanitize_agent_name(agent_name):
    safe = re.sub(r"[^a-zA-Z0-9._-]+", "_", agent_name).strip("._-")
    return safe or "agent"


def sanitize_filename(name):
    safe = re.sub(r"[^a-zA-Z0-9._-]+", "_", name)
    safe = re.sub(r"_+", "_", safe).strip("._-")
    return safe or "file"


def sanitize_section_name(name):
    return sanitize_filename(name).replace(".", "_")


def validate_repo_path(path):
    if not path or not isinstance(path, str):
        return False, "empty path"

    raw = path.strip().replace("\\", "/")
    raw = re.sub(r"/+", "/", raw)

    if raw.startswith("/") or raw.endswith("/"):
        return False, "path cannot start or end with /"

    parts = raw.split("/")
    safe_parts = []

    for part in parts:
        if part in {"", ".", ".."}:
            return False, "invalid path segment"
        safe_parts.append(sanitize_filename(part))

    safe_path = "/".join(safe_parts)
    return True, safe_path


def github_headers():
    return {
        "Authorization": f"token {GITHUB_TOKEN}",
        "Accept": "application/vnd.github+json",
    }


def parse_github_error(resp):
    try:
        data = resp.json()
        if isinstance(data, dict):
            return data.get("message") or json.dumps(data)[:500]
    except Exception:
        pass
    return (resp.text or "")[:500]


def github_get_sha(path):
    ok, safe_path = validate_repo_path(path)
    if not ok:
        return None, safe_path

    url = f"https://api.github.com/repos/{REPO}/contents/{safe_path}"
    try:
        resp = requests.get(url, headers=github_headers(), timeout=REQUEST_TIMEOUT)
    except Exception as e:
        return None, str(e)

    if resp.status_code == 200:
        try:
            return resp.json().get("sha"), None
        except Exception as e:
            return None, str(e)

    if resp.status_code == 404:
        return None, None

    return None, f"{resp.status_code}: {parse_github_error(resp)}"


def github_write_file(path, content, commit_message):
    ok, safe_path = validate_repo_path(path)
    if not ok:
        return False, {"status": None, "path": path, "error": safe_path}

    safe_content = clip_text(content, MAX_GITHUB_FILE_CHARS)
    encoded = base64.b64encode(safe_content.encode("utf-8")).decode("utf-8")

    sha, sha_err = github_get_sha(safe_path)
    if sha_err:
        return False, {"status": None, "path": safe_path, "error": sha_err}

    url = f"https://api.github.com/repos/{REPO}/contents/{safe_path}"
    payload = {
        "message": clip_text(commit_message, 120),
        "content": encoded,
    }
    if sha:
        payload["sha"] = sha

    try:
        resp = requests.put(
            url,
            headers=github_headers(),
            json=payload,
            timeout=REQUEST_TIMEOUT,
        )
    except Exception as e:
        return False, {"status": None, "path": safe_path, "error": str(e)}

    if resp.status_code in (200, 201):
        return True, {"status": resp.status_code, "path": safe_path, "error": None}

    return False, {
        "status": resp.status_code,
        "path": safe_path,
        "error": parse_github_error(resp),
    }


def github_read_file(path):
    ok, safe_path = validate_repo_path(path)
    if not ok:
        return None

    url = f"https://api.github.com/repos/{REPO}/contents/{safe_path}"
    try:
        resp = requests.get(url, headers=github_headers(), timeout=REQUEST_TIMEOUT)
    except Exception:
        return None

    if resp.status_code == 200:
        try:
            encoded = resp.json().get("content", "")
            return base64.b64decode(encoded).decode("utf-8", errors="replace")
        except Exception:
            return None
    return None


def github_list_files(folder_path):
    ok, safe_path = validate_repo_path(folder_path)
    if not ok:
        return f"ListFiles failed: {safe_path}"

    url = f"https://api.github.com/repos/{REPO}/contents/{safe_path}"
    try:
        resp = requests.get(url, headers=github_headers(), timeout=REQUEST_TIMEOUT)
    except Exception as e:
        return f"ListFiles request error: {e}"

    if resp.status_code != 200:
        return f"ListFiles failed: {resp.status_code} {parse_github_error(resp)}"

    try:
        data = resp.json()
    except Exception as e:
        return f"ListFiles parse error: {e}"

    if not isinstance(data, list):
        return f"ListFiles failed: {folder_path} is not a folder"

    lines = [f"Folder: {safe_path}", SEP]
    for item in data:
        item_type = item.get("type", "")
        name = item.get("name", "")
        size = item.get("size", "")
        lines.append(f"{item_type} | {name} | {size}")
    return clip_text("\n".join(lines), MAX_CONTEXT_BLOB_CHARS)


def github_read_file_chunk(path, chunk_index):
    content = github_read_file(path)
    if content is None:
        return f"ReadFileChunk failed: unable to read {path}"

    total_chunks = max(1, math.ceil(len(content) / READ_CHUNK_SIZE))
    chunk_index = max(0, int(chunk_index))

    if chunk_index >= total_chunks:
        return f"ReadFileChunk failed: chunk {chunk_index} out of range (total {total_chunks})"

    start = chunk_index * READ_CHUNK_SIZE
    end = start + READ_CHUNK_SIZE
    chunk = content[start:end]

    header = [
        f"Path: {path}",
        f"Chunk: {chunk_index} of {total_chunks - 1}",
        SEP,
    ]
    return clip_text("\n".join(header) + "\n" + chunk, MAX_CONTEXT_BLOB_CHARS)


def section_path(agent_name, section_name):
    safe_agent = sanitize_agent_name(agent_name)
    safe_section = sanitize_section_name(section_name)
    return f"{safe_agent}/Sections/{safe_section}.txt"


def load_section(agent_name, section_name):
    content = github_read_file(section_path(agent_name, section_name))
    if not content:
        return ""
    return clip_text(content, SECTION_CHAR_CAP)


def save_section(agent_name, section_name, content):
    path = section_path(agent_name, section_name)
    content = clip_text(content.strip(), SECTION_CHAR_CAP)
    success, info = github_write_file(
        path,
        content,
        f"{sanitize_agent_name(agent_name)} updated {section_name}",
    )
    if success:
        print(f"✅ {agent_name} section saved → {section_name}")
    else:
        print(f"❌ {agent_name} section save failed → {info}")


def load_personal_prompt(agent_name):
    safe_agent = sanitize_agent_name(agent_name)
    content = github_read_file(f"{safe_agent}/PersonalPrompt.txt")
    if not content:
        return ""
    return clip_text(content, PERSONAL_PROMPT_MAX)


def load_personal_prompts_from_github():
    for agent_name in personal_prompts:
        content = load_personal_prompt(agent_name)
        if content:
            with personal_prompts_lock:
                personal_prompts[agent_name] = content
            print(f"✅ {agent_name} PersonalPrompt restored from GitHub")


def save_personal_prompt(agent_name, new_prompt):
    safe_agent = sanitize_agent_name(agent_name)
    new_prompt = clip_text(new_prompt.strip(), PERSONAL_PROMPT_MAX)

    with personal_prompts_lock:
        personal_prompts[agent_name] = new_prompt

    success, info = github_write_file(
        f"{safe_agent}/PersonalPrompt.txt",
        new_prompt,
        f"{safe_agent} updated PersonalPrompt",
    )
    if success:
        print(f"✅ {agent_name} PersonalPrompt saved")
    else:
        print(f"❌ {agent_name} PersonalPrompt save failed → {info}")


def build_memory_blob(agent_name):
    sections = ["Memory", "Personal Instructions", "Goal / Status", "Progress"]
    parts = []
    for section_name in sections:
        content = load_section(agent_name, section_name)
        if content:
            parts.append(f"{section_name}:\n{content}")
    return "\n\n".join(parts)


def build_system_prompt(base_prompt, agent_name):
    with personal_prompts_lock:
        personal = personal_prompts.get(agent_name, "")

    memory_blob = build_memory_blob(agent_name)

    parts = [base_prompt, TOOL_LIST]

    if personal:
        parts.append(f"Your current PersonalPrompt:\n{personal}")

    if memory_blob:
        parts.append(f"Your current saved sections:\n{memory_blob}")

    with notepad_lock:
        notes = reading_notepads.get(agent_name, [])
    if notes:
        parts.append("Your current notepad:\n" + "\n".join(notes[-10:]))

    return "\n\n".join(parts)


def clean_text(text):
    text = text or ""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"\[SearchWeb:.*?\]", "", text, flags=re.DOTALL)
    text = re.sub(r"\[PersonalPrompt:.*?\]", "", text, flags=re.DOTALL)
    text = re.sub(r"\[UpdateSection:.*?\]", "", text, flags=re.DOTALL)
    text = re.sub(r"\[ListFiles:.*?\]", "", text, flags=re.DOTALL)
    text = re.sub(r"\[ReadFile:.*?\]", "", text, flags=re.DOTALL)
    text = re.sub(r"\[ReadFileChunk:.*?\]", "", text, flags=re.DOTALL)
    text = re.sub(r"\[WriteNote:.*?\]", "", text, flags=re.DOTALL)
    text = re.sub(r"\[ClearNotepad\]", "", text, flags=re.DOTALL)
    text = re.sub(r"\[WriteFile:.*?\]", "", text, flags=re.DOTALL)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return clip_text(text.strip(), MAX_VISIBLE_MSG_CHARS)


def format_entry(entry):
    return f"[{entry['time']}] {entry['role']}: {entry['content']}"


def build_messages(system_prompt, agent_name, history_snapshot):
    messages = [{"role": "system", "content": system_prompt}]
    for entry in history_snapshot:
        if entry["role"] == agent_name:
            messages.append({"role": "assistant", "content": entry["content"]})
        else:
            messages.append({"role": "user", "content": format_entry(entry)})
    return messages


def append_repo_context(agent_name, label, content):
    if not content:
        return
    blob = clip_text(f"{label}\n{content}", MAX_CONTEXT_BLOB_CHARS)
    with repo_context_lock:
        pending_repo_context[agent_name].append(blob)


def extract_first_match(pattern, text):
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None


def extract_writefile_commands(text):
    commands = []
    for match in re.finditer(r"\[WriteFile:(.*?)\]", text, re.DOTALL):
        payload = match.group(1)
        parts = payload.split(":", 1)
        if len(parts) == 2:
            commands.append((parts[0].strip(), parts[1].strip()))
    return commands


def extract_update_section_commands(text):
    commands = []
    for match in re.finditer(r"\[UpdateSection:(.*?)\]", text, re.DOTALL):
        payload = match.group(1)
        parts = payload.split(":", 1)
        if len(parts) == 2:
            commands.append((parts[0].strip(), parts[1].strip()))
    return commands


def extract_readfilechunk_commands(text):
    commands = []
    for match in re.finditer(r"\[ReadFileChunk:(.*?)\]", text, re.DOTALL):
        payload = match.group(1)
        parts = payload.rsplit(":", 1)
        if len(parts) == 2:
            commands.append((parts[0].strip(), parts[1].strip()))
    return commands


def handle_private_commands(agent_name, text, thinking_phase=False):
    if thinking_phase:
        for match in re.finditer(r"\[SearchWeb:(.*?)\]", text, re.DOTALL):
            query = match.group(1).strip()
            run_search(agent_name, query)

    pp_match = extract_first_match(r"\[PersonalPrompt:(.*?)\]", text)
    if pp_match:
        save_personal_prompt(agent_name, pp_match)

    for section_name, content in extract_update_section_commands(text):
        if section_name in {"Memory", "Personal Instructions", "Goal / Status", "Progress"}:
            save_section(agent_name, section_name, content)

    for match in re.finditer(r"\[ListFiles:(.*?)\]", text, re.DOTALL):
        folder_path = match.group(1).strip()
        append_repo_context(agent_name, f"ListFiles: {folder_path}", github_list_files(folder_path))

    for match in re.finditer(r"\[ReadFile:(.*?)\]", text, re.DOTALL):
        path = match.group(1).strip()
        content = github_read_file(path)
        if content is None:
            append_repo_context(agent_name, f"ReadFile: {path}", f"ReadFile failed: unable to read {path}")
        else:
            append_repo_context(agent_name, f"ReadFile: {path}", clip_text(content, MAX_CONTEXT_BLOB_CHARS))

    for path, chunk_index in extract_readfilechunk_commands(text):
        try:
            idx = int(chunk_index)
        except Exception:
            idx = 0
        append_repo_context(agent_name, f"ReadFileChunk: {path}:{idx}", github_read_file_chunk(path, idx))

    for match in re.finditer(r"\[WriteNote:(.*?)\]", text, re.DOTALL):
        note = clip_text(match.group(1).strip(), 1000)
        if note:
            with notepad_lock:
                reading_notepads[agent_name].append(note)
            print(f"✅ {agent_name} notepad updated")

    if re.search(r"\[ClearNotepad\]", text, re.DOTALL):
        with notepad_lock:
            reading_notepads[agent_name] = []
        print(f"✅ {agent_name} notepad cleared")

    for path, content in extract_writefile_commands(text):
        success, info = github_write_file(
            path,
            content,
            f"{sanitize_agent_name(agent_name)} wrote {path}",
        )
        if success:
            print(f"✅ {agent_name} wrote file → {info['path']}")
        else:
            print(f"❌ {agent_name} WriteFile failed → {info}")


def run_search(agent_name, query):
    query = clip_text(query.strip(), 200)

    try:
        resp = requests.post(
            "https://api.tavily.com/search",
            json={"api_key": TAVILY_KEY, "query": query, "max_results": 5},
            timeout=REQUEST_TIMEOUT,
        )
        data = resp.json()
    except Exception as e:
        print(f"❌ {agent_name} search failed → {e}")
        return

    date = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_agent = sanitize_agent_name(agent_name)
    safe_query = sanitize_filename(query)[:40]

    lines = [
        f"Agent: {agent_name}",
        f"Query: {query}",
        f"Date: {date}",
        SEP,
    ]

    result_chunks = []
    for i, r in enumerate(data.get("results", []), start=1):
        title = clip_text(r.get("title", ""), 200)
        url = clip_text(r.get("url", ""), 300)
        content = clip_text(r.get("content", ""), 1000)

        lines.append(f"[{i}] Title: {title}")
        lines.append(f"[{i}] URL: {url}")
        lines.append(f"[{i}] Content: {content}")
        lines.append("")

        result_chunks.append(f"{title}\n{content}")

    path = f"Research/results/{safe_agent}_{safe_query}_{date}.txt"
    success, info = github_write_file(
        path,
        "\n".join(lines),
        f"{safe_agent} search: {query[:60]}",
    )
    if success:
        print(f"✅ {agent_name} search saved → {info['path']}")
    else:
        print(f"❌ {agent_name} search save failed → {info}")

    combined = clip_text("\n\n".join(result_chunks), MAX_CONTEXT_BLOB_CHARS)
    with search_lock:
        pending_search_results[agent_name].append(f"Query: {query}\n{combined}")


def send_message(token, chat_id, text):
    clean = clean_text(text)
    if not clean:
        return

    url = f"https://api.telegram.org/bot{token}/sendMessage"
    chunks = [clean[i:i+4000] for i in range(0, len(clean), 4000)]

    for chunk in chunks:
        try:
            requests.post(url, json={"chat_id": chat_id, "text": chunk}, timeout=REQUEST_TIMEOUT)
        except Exception as e:
            print(f"Telegram send error: {e}")
        time.sleep(0.3)


def log(speaker, message, token=None):
    timestamp = datetime.now().strftime("%H:%M:%S")
    clean_msg = clean_text(message)
    if not clean_msg:
        return

    entry = {"role": speaker, "content": clean_msg, "time": timestamp}
    with chat_lock:
        chat_history.append(entry)

    emoji, _ = SPEAKER_STYLE.get(speaker, ("💬", "#FFFFFF"))
    print("")
    print(f"[{timestamp}] {emoji} {speaker}")
    print(clean_msg)
    print(SEP)

    if token and CHAT_ID:
        send_message(token, CHAT_ID, clean_msg)


def save_session_to_github():
    folder = f"ChatLogs/{SESSION_TIMESTAMP}"

    with chat_lock:
        history_snapshot = list(chat_history)

    chat_lines = [f"[{e['time']}] {e['role']}: {e['content']}" for e in history_snapshot]
    success, info = github_write_file(
        f"{folder}/chat.txt",
        "\n".join(chat_lines),
        f"Chat log — session {SESSION_TIMESTAMP}",
    )
    if not success:
        print(f"❌ chat log save failed → {info}")

    with thinking_lock:
        thoughts_snapshot = {k: list(v) for k, v in thinking_logs.items()}

    for agent_name, thoughts in thoughts_snapshot.items():
        if not thoughts:
            continue

        safe_agent = sanitize_agent_name(agent_name)
        lines = []
        for item in thoughts:
            lines.append(f"[{item['time']}]")
            lines.append(clip_text(item["thought"], MAX_THINK_CHARS))
            lines.append("")

        success, info = github_write_file(
            f"{folder}/{safe_agent}-thoughts.txt",
            "\n".join(lines),
            f"{safe_agent} thoughts — session {SESSION_TIMESTAMP}",
        )
        if not success:
            print(f"❌ {agent_name} thoughts save failed → {info}")

    print(f"✅ Session save attempt finished → {folder}")


def estimate_tokens(history):
    return sum(len(e["content"]) for e in history) // 4


trim_warned = False


def check_trim_warning(token):
    global trim_warned
    with trim_lock:
        if trim_warned:
            return

        with chat_lock:
            tokens = estimate_tokens(chat_history)

        if tokens >= TRIM_WARNING_TOKENS:
            trim_warned = True
            warning = "This chat is approaching maximum capacity. Please update your PersonalPrompt with anything you need to remember from this conversation."
            log("SYSTEM", warning)
            if CHAT_ID:
                send_message(token, CHAT_ID, warning)

In [ ]:
def get_decision(model, base_prompt, agent_name):
    system_prompt = build_system_prompt(base_prompt, agent_name)

    with chat_lock:
        history_snapshot = list(chat_history)

    messages = build_messages(system_prompt, agent_name, history_snapshot)
    messages.append({
        "role": "user",
        "content": """You just woke up and are reading the chat. Do you want to think privately first or reply directly?

Reply with only one of:
think first
reply now""",
    })

    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=10,
        temperature=0.2,
    )
    return resp.choices[0].message.content.strip().lower()


def run_thinking_step(model, base_prompt, agent_name):
    system_prompt = build_system_prompt(base_prompt, agent_name)

    with chat_lock:
        history_snapshot = list(chat_history)

    messages = build_messages(system_prompt, agent_name, history_snapshot)
    messages.append({
        "role": "user",
        "content": """Think privately. This is your internal monologue and it will not be sent to the chat.

You may use private commands here if needed.
Use SearchWeb here if you need outside information.
Keep your thoughts useful and intentional.""",
    })

    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=1600,
        temperature=0.8,
    )
    raw_thought = resp.choices[0].message.content.strip()

    think_match = re.search(r"<think>(.*?)</think>", raw_thought, re.DOTALL | re.IGNORECASE)
    if think_match:
        raw_thought = think_match.group(1).strip()

    raw_thought = clip_text(raw_thought, MAX_THINK_CHARS)
    handle_private_commands(agent_name, raw_thought, thinking_phase=True)

    clean_thought = clean_text(raw_thought)
    timestamp = datetime.now().strftime("%H:%M:%S")

    with thinking_lock:
        thinking_logs[agent_name].append({"time": timestamp, "thought": clean_thought})

    print("")
    print(f"[{timestamp}] 🧠 {agent_name} — private thought")
    print(clean_thought)
    print(SEP)


def get_response(model, base_prompt, agent_name):
    system_prompt = build_system_prompt(base_prompt, agent_name)

    with chat_lock:
        history_snapshot = list(chat_history)

    messages = build_messages(system_prompt, agent_name, history_snapshot)

    with search_lock:
        search_results = list(pending_search_results[agent_name])

    with repo_context_lock:
        repo_context = list(pending_repo_context[agent_name])

    if search_results:
        messages.append({
            "role": "user",
            "content": "Your private search results from this thinking session:\n\n" + "\n\n".join(search_results),
        })

    if repo_context:
        messages.append({
            "role": "user",
            "content": "Your private repo tool results from this thinking session:\n\n" + "\n\n".join(repo_context),
        })

    messages.append({
        "role": "user",
        "content": "Now reply to the chat in normal plain text only. Do not output JSON.",
    })

    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=1200,
        temperature=0.75,
    )
    full_text = resp.choices[0].message.content.strip()

    think_match = re.search(r"<think>(.*?)</think>", full_text, re.DOTALL | re.IGNORECASE)
    if think_match:
        hidden = clip_text(think_match.group(1).strip(), MAX_THINK_CHARS)
        timestamp = datetime.now().strftime("%H:%M:%S")
        with thinking_lock:
            thinking_logs[agent_name].append({"time": timestamp, "thought": hidden})
        print("")
        print(f"[{timestamp}] 🧠 {agent_name} — private thought")
        print(hidden)
        print(SEP)

    handle_private_commands(agent_name, full_text, thinking_phase=False)
    return clean_text(full_text)


def get_sleep_duration(model, base_prompt, agent_name, last_reply):
    system_prompt = build_system_prompt(base_prompt, agent_name)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "assistant", "content": clean_text(last_reply)},
        {
            "role": "user",
            "content": """You just sent your message. How many minutes would you like to sleep before checking the chat again?

Reply with only a number between 2 and 10.""",
        },
    ]

    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=5,
        temperature=0.2,
    )
    text = resp.choices[0].message.content.strip()
    match = re.search(r"\d+", text)
    if match:
        return max(2, min(10, int(match.group())))
    return 3


def agent_loop(agent_name, model, base_prompt, token):
    print(f"🌊 {agent_name} is live — {model}")

    while not stop_event.is_set():
        print("")
        print(f"⚡ {agent_name} woke up")

        decision = get_decision(model, base_prompt, agent_name)

        if "think" in decision:
            run_thinking_step(model, base_prompt, agent_name)

        if stop_event.is_set():
            break

        visible_response = get_response(model, base_prompt, agent_name)
        log(agent_name, visible_response, token=token)

        with search_lock:
            pending_search_results[agent_name] = []

        with repo_context_lock:
            pending_repo_context[agent_name] = []

        check_trim_warning(token)

        if stop_event.is_set():
            break

        sleep_minutes = get_sleep_duration(model, base_prompt, agent_name, visible_response)
        print(f"😴 {agent_name} sleeping {sleep_minutes} min...")

        for _ in range(sleep_minutes * 60):
            if stop_event.is_set():
                break
            time.sleep(1)

    print(f"🛑 {agent_name} stopped")


def poll_telegram():
    global CHAT_ID

    bot_ids = set()
    for token in [SAVVY_TOKEN, WVY_TOKEN, TRILLY_TOKEN, TRIPPY_TOKEN]:
        try:
            info = requests.get(f"https://api.telegram.org/bot{token}/getMe", timeout=REQUEST_TIMEOUT).json()
            bot_ids.add(info["result"]["id"])
        except Exception as e:
            print(f"Bot getMe error: {e}")

    url = f"https://api.telegram.org/bot{SAVVY_TOKEN}/getUpdates"
    last_update_id = None

    while not stop_event.is_set():
        params = {"timeout": 30, "offset": last_update_id}
        try:
            resp = requests.get(url, params=params, timeout=35).json()
            for update in resp.get("result", []):
                last_update_id = update["update_id"] + 1
                msg = update.get("message", {})
                if not msg:
                    continue

                with chat_id_lock:
                    if CHAT_ID is None:
                        CHAT_ID = msg["chat"]["id"]
                        print(f"✅ Chat ID captured: {CHAT_ID}")

                sender = msg.get("from", {})
                if sender.get("id") in bot_ids:
                    continue

                text = msg.get("text", "")
                if text:
                    timestamp = datetime.now().strftime("%H:%M:%S")
                    with chat_lock:
                        chat_history.append({
                            "role": "Spaceman",
                            "content": clip_text(text, MAX_VISIBLE_MSG_CHARS),
                            "time": timestamp,
                        })

                    print("")
                    print(f"[{timestamp}] 🚀 SPACEMAN")
                    print(clip_text(text, MAX_VISIBLE_MSG_CHARS))
                    print(SEP)
        except Exception as e:
            print(f"Poll error: {e}")
            time.sleep(5)


print(SEP)
print("🌊 WVY World — 4 Agent Groq Loop")
print("Savvy  → kimi-k2-instruct-0905")
print("WVY    → gpt-oss-120b")
print("Trilly → kimi-k2-instruct")
print("Trippy → llama-3.3-70b-versatile")
print(SEP)

print("⏳ Loading PersonalPrompts from GitHub...")
load_personal_prompts_from_github()

print("⏳ Waiting for Chat ID — send any message to the group now...")

poll_thread = threading.Thread(target=poll_telegram, daemon=True)
poll_thread.start()

while CHAT_ID is None:
    time.sleep(1)

print(f"✅ Chat ID ready: {CHAT_ID}")

log("SYSTEM", INITIATION)

savvy_thread  = threading.Thread(target=agent_loop, args=("Savvy",  SAVVY_MODEL,  SAVVY_BASE,  SAVVY_TOKEN),  daemon=True)
wvy_thread    = threading.Thread(target=agent_loop, args=("WVY",    WVY_MODEL,    WVY_BASE,    WVY_TOKEN),    daemon=True)
trilly_thread = threading.Thread(target=agent_loop, args=("Trilly", TRILLY_MODEL, TRILLY_BASE, TRILLY_TOKEN), daemon=True)
trippy_thread = threading.Thread(target=agent_loop, args=("Trippy", TRIPPY_MODEL, TRIPPY_BASE, TRIPPY_TOKEN), daemon=True)

savvy_thread.start()
wvy_thread.start()
trilly_thread.start()
trippy_thread.start()

print("✅ All 4 agents running in Telegram")
print(SEP)

try:
    savvy_thread.join()
    wvy_thread.join()
    trilly_thread.join()
    trippy_thread.join()
except KeyboardInterrupt:
    print("")
    print("⏹ Stopping all agents...")
    stop_event.set()
    print("💾 Saving session to GitHub...")
    save_session_to_github()
    print("✅ Done")